# Imports lib

In [1]:
import os
import cv2
import json
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models

In [ ]:
# ============================
# 1. define paths and parameters
# ============================
base_dir = "D:/BDSL/V_data/W1-2"
video_files = [f for f in os.listdir(base_dir) if f.endswith(".mp4")]
annotation_file = os.path.join(base_dir, "output1.json")
output_dir = "BDSL_dataset/train"

In [ ]:
# ============================
# 2. Load annotations
# ============================
with open(annotation_file, "r", encoding="utf-8") as f:
    annotations = json.load(f)


In [ ]:
# ============================
# 3. Build segments from annotations
# ============================
def build_segments(annotation_data):
    """Normalize annotations to a list of {video, label, start_frame, end_frame}."""
    segments = []

    # Format B: nested output1.json structure
    if isinstance(annotation_data, dict):
        for label, users_block in annotation_data.items():
            if not isinstance(users_block, dict):
                continue
            for hands_block in users_block.values():
                if not isinstance(hands_block, dict):
                    continue
                for clips_block in hands_block.values():
                    if not isinstance(clips_block, dict):
                        continue
                    for clip_key, clip_info in clips_block.items():
                        if not isinstance(clip_info, dict):
                            continue
                        file_name = clip_info.get("FileName", clip_key)
                        trials = clip_info.get("trials", {})
                        if not isinstance(trials, dict):
                            continue
                        for trial in trials.values():
                            if not isinstance(trial, dict):
                                continue
                            start = trial.get("starting")
                            end = trial.get("ending")
                            if start is None or end is None:
                                continue
                            segments.append(
                                {
                                    "video": f"{file_name}.mp4",
                                    "label": str(label),
                                    "start_frame": int(start),
                                    "end_frame": int(end),
                                }
                            )

    return segments

segments = build_segments(annotations)


In [ ]:
# ============================
# 4. process videos and save frames
# ============================
# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Process each video
for video in video_files:
    video_path = os.path.join(base_dir, video)
    cap = cv2.VideoCapture(video_path)

    # Find normalized segments for this video
    video_segments = [seg for seg in segments if seg["video"] == video]

    for seg in video_segments:
        label = seg["label"]
        start_frame = seg["start_frame"]
        end_frame = seg["end_frame"]

        # Create label folder
        label_dir = os.path.join(output_dir, label)
        os.makedirs(label_dir, exist_ok=True)

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        frame_id = start_frame
        while frame_id <= end_frame:
            ret, frame = cap.read()
            if not ret:
                break

            # Resize frame
            frame = cv2.resize(frame, (224, 224))

            # Save frame
            frame_filename = os.path.join(label_dir, f"{video}_{frame_id}.jpg")
            cv2.imwrite(frame_filename, frame)

            frame_id += 1

    cap.release()

print("Frame extraction complete for all videos!")

Frame extraction complete for all videos!


In [ ]:
# ============================
# 5. Sequence Dataset (CNN+LSTM)
# ============================
class SignLanguageSequenceDataset(Dataset):
    def __init__(self, root_dir, sequence_length=16, stride=None, transform=None, extensions=(".jpg", ".png")):
        self.root_dir = root_dir
        self.sequence_length = int(sequence_length)
        self.stride = int(stride) if stride is not None else int(sequence_length)
        self.transform = transform
        self.extensions = tuple(ext.lower() for ext in extensions)

        # Use only valid label directories.
        self.labels = sorted(
            [entry.name for entry in os.scandir(root_dir) if entry.is_dir()]
        )
        self.label_to_idx = {label: idx for idx, label in enumerate(self.labels)}
        self.samples = []

        for label in self.labels:
            label_dir = os.path.join(root_dir, label)

            # Faster than repeated os.listdir + os.path.join in Python loops.
            frame_paths = sorted(
                [
                    entry.path
                    for entry in os.scandir(label_dir)
                    if entry.is_file() and entry.name.lower().endswith(self.extensions)
                ]
            )

            # Build fixed-length sequences with configurable stride.
            max_start = len(frame_paths) - self.sequence_length + 1
            if max_start <= 0:
                continue

            for start_idx in range(0, max_start, self.stride):
                seq_paths = frame_paths[start_idx:start_idx + self.sequence_length]
                if len(seq_paths) == self.sequence_length:
                    self.samples.append((seq_paths, self.label_to_idx[label]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        frame_paths, label_idx = self.samples[idx]
        frames = []
        for path in frame_paths:
            img = Image.open(path).convert("RGB")
            if self.transform is not None:
                img = self.transform(img)
            frames.append(img)

        # [seq_len, C, H, W]
        frames = torch.stack(frames, dim=0)
        return frames, label_idx

In [ ]:
# ============================
# 6. CNN+LSTM Model
# ============================
class CNN_LSTM(nn.Module):
    def __init__(self, num_classes, hidden_size=256, num_layers=1):
        super(CNN_LSTM, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        self.feature_dim = resnet.fc.in_features
        self.lstm = nn.LSTM(self.feature_dim, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        batch_size, seq_len, C, H, W = x.size()
        features = []
        for t in range(seq_len):
            f = self.cnn(x[:, t, :, :, :])
            f = f.view(batch_size, -1)
            features.append(f)
        features = torch.stack(features, dim=1)
        lstm_out, _ = self.lstm(features)
        out = lstm_out[:, -1, :]
        return self.fc(out)

In [ ]:
# ============================
# 7. Training Setup
# ============================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_dataset = SignLanguageSequenceDataset("BDSL_dataset/train", sequence_length=16, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

num_classes = len(train_dataset.label_to_idx)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN_LSTM(num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

c:\Users\prome\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\prome\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
# ============================
# 8. Training Loop with Best Model Saving
# ============================
def train_model(model, train_loader, criterion, optimizer, num_epochs=10, save_path="bdsl_best_model.pth"):
    best_loss = float('inf')
    best_epoch = 0
    
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for frames, labels in train_loader:
            frames, labels = frames.to(device), labels.to(device)
            outputs = model(frames)
            loss = criterion(outputs, labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0); correct += (predicted == labels).sum().item()
        
        avg_loss = running_loss / len(train_loader)
        acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Train Acc: {acc:.2f}%")
        
        # Save best model
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_epoch = epoch + 1
            torch.save(model.state_dict(), save_path)
            print(f"  ✓ Best model saved (Loss: {avg_loss:.4f})")
    
    print(f"\nTraining complete! Best model saved at epoch {best_epoch} with loss {best_loss:.4f}")
    return model


In [ ]:
# ============================
# 9. Train and Save Best Model
# ============================
trained_model = train_model(model, train_loader, criterion, optimizer, num_epochs=15, save_path="bdsl_best_cnn_lstm_model.pth")
print("Training complete! Best model has been saved as 'bdsl_best_cnn_lstm_model.pth'")

Epoch [1/15], Loss: 0.4799, Train Acc: 83.26%
  ✓ Best model saved (Loss: 0.4799)
